ratings1 и ratings2 — таблицы с данными о выставленных пользователями оценках фильмов. Они имеют одинаковую структуру и типы данных — на самом деле это две части одной таблицы с оценками фильмов.
userId — уникальный идентификатор пользователя, который выставил оценку;
movieId — уникальный идентификатор фильма;
rating — рейтинг фильма.

dates — таблица с датами выставления всех оценок.
date — дата и время выставления оценки фильму.

movies — таблица с информацией о фильмах.
movieId — уникальный идентификатор фильма;
title — название фильма и год его выхода;
genres — жанры фильма.

In [ ]:
import pandas as pd

movies_df = pd.read_csv('movies_data/ratings_movies.csv')

In [ ]:
display(movies_df.info())

In [ ]:
import pandas as pd
import re

# Загрузка файла
file_path = 'ratings_movies.csv'


# Функция для извлечения года выпуска из названия фильма
def get_year_release(arg):
    # Находим все слова по шаблону "(DDDD)"
    candidates = re.findall(r'\(\d{4}\)', arg)
    # Проверяем число вхождений
    if len(candidates) > 0:
        # Если число вхождений больше 0, очищаем строку от знаков "(" и ")"
        year = candidates[0].replace('(', '').replace(')', '')
        return int(year)
    else:
        # Если год не указан, возвращаем None
        return None

# Применяем функцию к столбцу 'title'
movies_df['year_release'] = movies_df['title'].apply(get_year_release)

# Подсчитываем количество фильмов без указания года выпуска
missing_year_count = movies_df['year_release'].isnull().sum()

# Вывод информации о новом столбце и количестве фильмов без года выпуска
print(movies_df[['title', 'year_release']].head())
print("Количество фильмов без указания года выпуска:", missing_year_count)


In [ ]:
# Фильтрация фильмов, выпущенных в 1999 году
movies_df_mean = movies_df[movies_df['year_release'].apply(str).str.contains("1999", regex=False, na=False, case=False)]

# Выполнено 1 агрегирование с группированием по столбцу 'title'
movies_df_mean = movies_df_mean.groupby(['title']).agg(rating_mean=('rating', 'mean')).reset_index()

# Сортировка по столбцу: 'rating_mean' (по возрастанию)
movies_df_mean = movies_df_mean.sort_values(['rating_mean'])

# Сброс индексов
movies_df_mean.reset_index(drop=True, inplace=True)

# Вывод только первого значения в столбце "title"
print(movies_df_mean.loc[0, 'title'])

# Вывод первых нескольких строк DataFrame
display(movies_df_mean[['title', 'rating_mean']].head())

In [ ]:
# Фильтрация фильмов, выпущенных в 2010 году
movies_2010 = movies_df[movies_df['year_release'] == 2010]

# Выполнение агрегирования с группированием по столбцу 'genres'
genres_mean_ratings = movies_2010.groupby('genres').agg(rating_mean=('rating', 'mean')).reset_index()

# Сортировка по столбцу 'rating_mean' (по возрастанию)
genres_mean_ratings = genres_mean_ratings.sort_values('rating_mean')

# Сброс индексов
genres_mean_ratings.reset_index(drop=True, inplace=True)

# Вывод сочетания жанров с наименьшей средней оценкой
print(genres_mean_ratings.loc[0, 'genres'])

In [ ]:
# Группировка данных по 'userId' и 'genres', и подсчет уникальных комбинаций
unique_genres_per_user = movies_df.groupby('userId')['genres'].nunique()

# Нахождение пользователя с наибольшим количеством уникальных комбинаций жанров
max_unique_genres_user = unique_genres_per_user.idxmax()

# Вывод идентификатора пользователя
print(max_unique_genres_user)

In [ ]:
# Подсчет количества оценок для каждого пользователя
user_ratings_count = movies_df.groupby('userId').size()

# Найдем минимальное количество оценок
min_ratings = user_ratings_count.min()

# Найдем пользователей с минимальным количеством оценок
users_with_min_ratings = user_ratings_count[user_ratings_count == min_ratings].index

# Рассчитаем среднюю оценку для этих пользователей
user_mean_ratings = movies_df[movies_df['userId'].isin(users_with_min_ratings)].groupby('userId')['rating'].mean()

# Найдем пользователя с наибольшей средней оценкой
user_with_highest_mean_rating = user_mean_ratings.idxmax()

# Вывод идентификатора пользователя
print(user_with_highest_mean_rating)

In [4]:
# Фильтрация фильмов, выпущенных в 2018 году
movies_2018 = movies_df[movies_df['year_release'] == 2018]

# Группировка по жанрам и вычисление среднего рейтинга и количества оценок
genres_ratings = movies_2018.groupby('genres').agg(
    rating_mean=('rating', 'mean'),
    rating_count=('rating', 'count')
).reset_index()

# Фильтрация по количеству оценок > 10
genres_ratings_filtered = genres_ratings[genres_ratings['rating_count'] > 10]

# Поиск жанра с наибольшим средним рейтингом
top_genre = genres_ratings_filtered.sort_values('rating_mean', ascending=False).iloc[0]

# Вывод сочетания жанров с наибольшим средним рейтингом
print("Сочетание жанров:", top_genre['genres'])
print("Средний рейтинг:", top_genre['rating_mean'])


Сочетание жанров: Action|Adventure|Sci-Fi
Средний рейтинг: 3.9285714285714284


In [7]:
import pandas as pd

# Добавляем новый признак year_rating
movies_df['year_rating'] = pd.to_datetime(movies_df['date']).dt.year

# Создаем сводную таблицу
pivot_table = movies_df.pivot_table(
    values='rating',  # Средний рейтинг
    index='year_rating',  # Год выставления оценки
    columns='genres',  # Жанры
    aggfunc='mean'  # Функция агрегации
)

# Вывод сводной таблицы
# print(pivot_table)
display(pivot_table)


genres,(no genres listed),Action,Action|Adventure,Action|Adventure|Animation,Action|Adventure|Animation|Children,Action|Adventure|Animation|Children|Comedy,Action|Adventure|Animation|Children|Comedy|Fantasy,Action|Adventure|Animation|Children|Comedy|IMAX,Action|Adventure|Animation|Children|Comedy|Romance,Action|Adventure|Animation|Children|Comedy|Sci-Fi,...,Romance|Thriller,Romance|War,Romance|Western,Sci-Fi,Sci-Fi|IMAX,Sci-Fi|Thriller,Sci-Fi|Thriller|IMAX,Thriller,War,Western
year_rating,,,,,,,,,,,,,,,,,,,,,
1996,NaN,2.730769,3.454545,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,2.666667,NaN,3.838095,NaN,3.117647
1997,NaN,3.538462,4.150000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,3.400000,NaN,3.923077,NaN,3.000000
1998,NaN,NaN,4.200000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.800000,NaN,NaN
1999,NaN,NaN,4.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2.000,NaN,NaN,NaN,NaN,4.000000,NaN,3.700000,4.5,4.000000
2000,NaN,2.588235,3.738462,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4.000,NaN,3.0,3.416667,NaN,2.142857,NaN,3.087912,3.0,4.058824
2001,NaN,3.000000,3.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,3.0,2.500000,NaN,2.500000,NaN,3.477273,3.0,3.111111
2002,NaN,2.750000,4.304348,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.000,NaN,NaN,3.750000,NaN,3.600000,NaN,3.583333,3.5,3.000000
2003,NaN,3.833333,3.277778,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.375,2.5,NaN,2.333333,NaN,3.142857,NaN,3.250000,3.0,4.000000
2004,NaN,2.700000,4.136364,NaN,NaN,4.000000,NaN,NaN,NaN,NaN,...,3.000,3.0,3.5,2.125000,NaN,NaN,NaN,3.464286,3.0,3.800000
